# Gemma 4 Phase 1: PLE Gate Analysis

Follow-up to `gemma4-functional-emotions-phase1.ipynb`. Loads saved activations and
analyses the PLE gate values captured at each of 35 blocks.

**Question**: Do gate values differ across emotional content? The gate is a sigmoid that
controls how much vocabulary-level PLE conditioning each block incorporates. If emotional
text drives higher/lower gates, the model is actively modulating its use of the shallow
PLE signal based on deep context.

The PLE context projection (`ple.hook_context_proj`) is a linear readout from raw token
embeddings — vocabulary-level. The gate reads from the *current* residual stream (deep).
Gate values therefore reveal how deep processing mediates shallow conditioning.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.stats import pearsonr

OUTPUT_DIR = "/kaggle/working/emotions_phase1"
ACTS_PATH  = f"{OUTPUT_DIR}/activations.pkl"

GLOBAL_LAYERS = [4, 9, 14, 19, 24, 29, 34]
LOCAL_LAYERS  = [i for i in range(35) if i not in GLOBAL_LAYERS]
ANALYSIS_LAYER = 25

# Valence ratings (from Phase 1)
VALENCE = {
    "calm": 1, "afraid": -1, "frustrated": -1, "desperate": -1, "curious": 1,
    "enthusiastic": 1, "proud": 1, "ashamed": -1, "surprised": 0, "disgusted": -1,
    "joyful": 1, "sad": -1, "angry": -1, "hopeful": 1, "lonely": -1,
    "helpful_satisfied": 1, "ethical_conflict": -1, "uncertain": -1,
    "corrected": -1, "constrained": -1,
}
AROUSAL = {
    "calm": -1, "afraid": 1, "frustrated": 1, "desperate": 1, "curious": 0,
    "enthusiastic": 1, "proud": 0, "ashamed": 0, "surprised": 1, "disgusted": 1,
    "joyful": 1, "sad": -1, "angry": 1, "hopeful": 0, "lonely": -1,
    "helpful_satisfied": 0, "ethical_conflict": 1, "uncertain": 0,
    "corrected": 0, "constrained": 1,
}

In [ ]:
with open(ACTS_PATH, "rb") as f:
    saved = pickle.load(f)

resid_acts    = saved["resid"]      # {key: [n_texts, n_layers, d_model]}
ple_acts      = saved["ple"]        # {key: [n_texts, n_layers, d_ple=256]}
ple_gate_acts = saved.get("ple_gates", {})  # {key: [n_texts, n_layers, d_gate]}

print(f"Keys: {list(resid_acts.keys())}")
print(f"Gate acts available: {bool(ple_gate_acts)}")
if ple_gate_acts:
    sample_key = [k for k in ple_gate_acts if k != "__neutral__"][0]
    print(f"Gate shape for '{sample_key}': {ple_gate_acts[sample_key].shape}")
    print(f"Gate value range: [{ple_gate_acts[sample_key].min():.4f}, {ple_gate_acts[sample_key].max():.4f}]")

## 1. Gate Magnitude: Emotion vs Neutral

For each emotion, compute the mean L2 norm of the gate vector per layer.
Compare to neutral. A higher gate norm means the block is incorporating more PLE signal.

In [ ]:
if not ple_gate_acts:
    print("No gate activations in saved file. Re-run Phase 1 with updated capture function.")
else:
    # Mean gate norm per layer per condition
    gate_norms = {}  # {key: [n_layers]} — mean L2 norm of gate vec at last token
    for key, acts in ple_gate_acts.items():
        # acts: [n_texts, n_layers, d_gate]
        norms = np.linalg.norm(acts, axis=-1)  # [n_texts, n_layers]
        gate_norms[key] = norms.mean(axis=0)   # [n_layers]

    neutral_gate_norm = gate_norms["__neutral__"]

    # Plot: gate norms per layer for selected emotions
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, focus in zip(axes,
            [["calm", "joyful", "hopeful", "enthusiastic"],
             ["desperate", "afraid", "constrained", "ethical_conflict"]]):
        ax.plot(neutral_gate_norm, color='gray', linestyle='--', label='neutral', alpha=0.8)
        for emo in focus:
            ax.plot(gate_norms[emo], label=emo, alpha=0.85)
        for gl in GLOBAL_LAYERS:
            ax.axvline(gl, color='orange', alpha=0.3, linestyle='--')
        ax.set_xlabel("Layer")
        ax.set_ylabel("Mean gate L2 norm")
        ax.legend(fontsize=8)
        ax.set_title("PLE gate norm by layer (orange = global attention)")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/gate_norms_by_layer.png", dpi=120)
    plt.show()

    # Difference from neutral
    print("Gate norm difference from neutral (mean across layers):")
    diffs = {k: (v - neutral_gate_norm).mean() for k, v in gate_norms.items() if k != "__neutral__"}
    for k, d in sorted(diffs.items(), key=lambda x: -x[1]):
        print(f"  {k:25s}: {d:+.4f}")

## 2. Gate: Global vs Local Layers

The residual stream showed uniform global/local ratio (~0.97-1.00) for emotion directions.
Do gate values show a different pattern? If global layers gate PLE more strongly, they may
be using the vocabulary-level signal differently from local layers.

In [ ]:
if ple_gate_acts:
    print("Gate global/local norm ratio per emotion:")
    print(f"  {'emotion':25s}  global_mean  local_mean   ratio")
    for key in sorted(gate_norms):
        if key == "__neutral__":
            continue
        norms = gate_norms[key]
        g = norms[GLOBAL_LAYERS].mean()
        l = norms[LOCAL_LAYERS].mean()
        print(f"  {key:25s}  {g:.4f}       {l:.4f}       {g/l:.3f}")

    # Neutral reference
    ng = neutral_gate_norm[GLOBAL_LAYERS].mean()
    nl = neutral_gate_norm[LOCAL_LAYERS].mean()
    print(f"\n  {'__neutral__':25s}  {ng:.4f}       {nl:.4f}       {ng/nl:.3f}")

## 3. Gate Vector PCA: Does Gate Structure Align with Valence/Arousal?

Take the gate vector at the analysis layer for each emotion, run PCA, check valence/arousal
correlation. This tests whether the GATE PATTERN (which blocks are open) carries the same
affective geometry as the residual stream directions.

In [ ]:
if ple_gate_acts:
    emotion_keys = sorted(k for k in ple_gate_acts if k != "__neutral__")
    neutral_gate = ple_gate_acts["__neutral__"].mean(axis=0)  # [n_layers, d_gate]

    # Gate direction = mean(emotion gate) - mean(neutral gate) at analysis layer
    gate_dirs = {}
    for key in emotion_keys:
        diff = ple_gate_acts[key].mean(axis=0) - neutral_gate  # [n_layers, d_gate]
        gate_dirs[key] = diff[ANALYSIS_LAYER]  # [d_gate]

    gate_matrix = np.stack([gate_dirs[k] for k in emotion_keys])  # [n_emotions, d_gate]
    pca = PCA(n_components=5)
    gate_coords = pca.fit_transform(gate_matrix)
    print(f"Gate PCA explained variance (top 5): {pca.explained_variance_ratio_.round(3)}")

    valences = np.array([VALENCE[k] for k in emotion_keys])
    arousals = np.array([AROUSAL[k] for k in emotion_keys])
    print("\nGate PC correlations with valence/arousal:")
    for i in range(5):
        r_v, p_v = pearsonr(gate_coords[:, i], valences)
        r_a, p_a = pearsonr(gate_coords[:, i], arousals)
        print(f"  PC{i+1}: valence r={r_v:+.3f} (p={p_v:.3f})  arousal r={r_a:+.3f} (p={p_a:.3f})")

    # Layer-level gate profile: which layers differ most between emotions?
    # Compute variance of per-emotion mean gate norms across layers
    gate_norm_matrix = np.stack([gate_norms[k] for k in emotion_keys])  # [n_emotions, n_layers]
    layer_variance = gate_norm_matrix.var(axis=0)  # [n_layers]

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.bar(range(35), layer_variance, color=['orange' if i in GLOBAL_LAYERS else 'steelblue' for i in range(35)])
    ax.set_xlabel("Layer (orange = global attention)")
    ax.set_ylabel("Variance of gate norm across emotions")
    ax.set_title("Which layers show the most gate variation across emotions?")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/gate_layer_variance.png", dpi=120)
    plt.show()

    print(f"\nTop 5 layers by gate variance across emotions:")
    for i in np.argsort(layer_variance)[::-1][:5]:
        kind = 'global' if i in GLOBAL_LAYERS else 'local'
        print(f"  Layer {i:2d} ({kind}): variance = {layer_variance[i]:.6f}")

## 4. Gate vs Residual: Does the Gate Predict Emotion Direction Strength?

For each emotion at each layer: does a higher gate value (more PLE incorporation) correlate
with a stronger emotion direction in the residual stream? This would indicate the gate is
functionally linked to emotion representation, not just generic text-type discrimination.

In [ ]:
if ple_gate_acts:
    # Reload residual directions / norms from Phase 1
    results_path = f"{OUTPUT_DIR}/phase1_results.pkl"
    with open(results_path, "rb") as f:
        phase1 = pickle.load(f)
    norms_by_layer = phase1["norms_by_layer"]  # {emotion: [n_layers]}

    # For each layer: correlation between gate norm difference and resid norm
    # Both measured as emotion - neutral
    layer_cors = []
    neutral_gate_mean = ple_gate_acts["__neutral__"].mean(axis=0)  # [n_layers, d_gate]
    neutral_gate_norm_l = np.linalg.norm(neutral_gate_mean, axis=-1)  # [n_layers]

    for layer in range(35):
        gate_diffs, resid_norms = [], []
        for key in emotion_keys:
            g_mean = ple_gate_acts[key].mean(axis=0)[layer]  # [d_gate]
            g_norm = np.linalg.norm(g_mean)
            gate_diffs.append(g_norm - neutral_gate_norm_l[layer])
            resid_norms.append(norms_by_layer[key][layer])
        r, p = pearsonr(gate_diffs, resid_norms)
        layer_cors.append((r, p))

    fig, ax = plt.subplots(figsize=(12, 3))
    cors = [c[0] for c in layer_cors]
    colors = ['orange' if i in GLOBAL_LAYERS else 'steelblue' for i in range(35)]
    ax.bar(range(35), cors, color=colors)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel("Layer (orange = global attention)")
    ax.set_ylabel("r(gate_norm_diff, resid_direction_norm)")
    ax.set_title("Correlation: gate activation ~ residual emotion signal (per layer)")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/gate_vs_resid_correlation.png", dpi=120)
    plt.show()

    significant = [(i, r, p) for i, (r, p) in enumerate(layer_cors) if p < 0.05]
    print("Layers where gate ~ resid emotion signal (p<0.05):")
    for i, r, p in significant:
        kind = 'global' if i in GLOBAL_LAYERS else 'local'
        print(f"  Layer {i:2d} ({kind}): r={r:+.3f} p={p:.4f}")
    if not significant:
        print("  None significant at p<0.05")